# [LEC] 캡스톤 프로젝트: 리서치 자동화 멀티 에이전트 시스템

슬라이드와 설명, 퀴즈로 캡스톤에서 만들 시스템의 구조와 평가 기준을 익힙니다.

셀을 위에서부터 차례로 실행하면 슬라이드와 퀴즈가 나타납니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. Workflow 구성 (슬라이드) (퀴즈)
  - 1.1 전체 흐름 (슬라이드) (퀴즈)
  - 1.2 Human-in-the-loop (슬라이드) (퀴즈)
- 2. MCP와 RAGaaS 활용 (슬라이드) (퀴즈)
- 3. Deep Agents와 Langfuse (슬라이드) (퀴즈)
  - 3.1 Deep Agents (슬라이드) (퀴즈)
  - 3.2 Langfuse (슬라이드) (퀴즈)
- 4. 보고서 품질 평가 (슬라이드) (퀴즈)
- 5. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("4-3", [1, 2])

이번 시간은 4일 동안 배운 기술을 한 프로그램에 모으는 캡스톤입니다. 만들 것은 **리서치 자동화 보고서 작성 Multi-Agent 시스템**입니다.

1. Workflow 구성: 전체 흐름, Human-in-the-loop
2. MCP와 RAGaaS 활용
3. Deep Agents와 Langfuse
4. 보고서 품질 평가

**왜 배우나:** LangGraph 멀티 에이전트, 사람 승인(Human-in-the-loop), MCP, RAGaaS, Deep Agents의 Skill, Langfuse 관측과 평가를 따로따로 익혔습니다. 실제 업무용 Agent는 이것들을 함께 씁니다. 각 기술이 전체 흐름의 어느 단계를 맡는지 알아야 AI에게 일을 나눠 시키고 결과를 검토할 수 있습니다.

<details><summary>강사용 메모</summary>

| 항목 | 내용 |
|---|---|
| 버전 | C (슬라이드 요약 + 내용 있는 곳만 "왜 배우나") |
| 원본 PDF | `course/Chapter02/slides/day02/4-3-research-automation-multi-agent-system.pdf` (9쪽. 표지 p1, 본문 p2~p9, 글만 있는 슬라이드) |
| 원본 코드 | 이 노트북에는 없습니다. 실습은 `4-3_capstone_LAB.ipynb`(안내)와 `course/Chapter02/capstone/` 폴더에서 합니다. |
| API 호출 | 없습니다. 슬라이드 준비 셀만 실행하면 됩니다. |

---

- 커리큘럼 4.3(4일차 4.0H): "4일 학습요소 통합 구현 (자유 주제 혹은 스켈레톤 코드 기반 개발 중 선택)". 블록 안 시간 배분은 커리큘럼에 없습니다. 목차 단계에서 LEC 20분 + 안내 노트북·준비 20분 + 개발·실행·정답 비교 약 200분을 가정했습니다.
- 슬라이드에는 배점이 없습니다. 평가는 슬라이드의 LLM Judge 네 항목 + 안내 노트북 체크리스트 7단계로 합니다(사용자 결정). 받은 zip README의 배점 표(Streamlit UI 15점 포함)는 옮기지 않았고, Streamlit은 조기 완료자 선택 과제로만 둡니다.
- 11차수 피드백: 캡스톤 시간 부족, 3일차에 미리 예고 요청. 안내 노트북 1·2장을 3일차 예고에 쓸 수 있습니다.

</details>

## 1. Workflow 구성 <small>(슬라이드) (퀴즈)</small>

### 1.1 전체 흐름 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-3", [3])

전체 흐름은 여섯 단계입니다.

1. 사용자가 주제를 입력합니다.
2. Supervisor가 Researcher 실행을 결정합니다.
3. Researcher가 웹 검색, RAGaaS, MCP 문서 검색으로 자료를 모읍니다.
4. 사용자가 리서치 결과를 확인하고 보고서 작성을 승인합니다.
5. Writer가 보고서 초안을 작성합니다.
6. 최종 보고서를 평가하고 결과를 출력합니다.

Supervisor가 다음 차례를 정하고, Researcher와 Writer가 일을 나눠 맡는 구조는 앞에서 본 Supervisor 패턴 그대로입니다. 여기에 사람의 승인(4단계)과 평가(6단계)가 더해집니다.

**왜 배우나:** 캡스톤의 개발 단계도 이 흐름을 따라갑니다. 지금 고치는 코드가 여섯 단계 중 어디인지 알면 AI에게 줄 지시의 범위를 좁힐 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/4-3/q11.json")

**정리:** 주제 입력 → Supervisor 결정 → Researcher 자료 수집 → 사용자 승인 → Writer 초안 → 평가·출력의 여섯 단계입니다.

<details><summary>강사용 메모</summary>

- 코드와 대응: ② Supervisor 노드(구조화 출력 `RouteDecision`: researcher/writer/END) ③ Researcher 노드(`search_web`·`rag_search`·MCP 도구 3개) ④ `interrupt_before=['writer']` + 터미널 `input()` ⑤ Writer 노드 ⑥ `evaluate_report`(LLM Judge) + 점수 막대 출력.
- 웹 검색 `search_web`은 정답에서도 실제 검색이 아니라 정해 둔 글 네 줄을 돌려주는 흉내 도구입니다.

</details>

### 1.2 Human-in-the-loop <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-3", [4])

Human-in-the-loop는 자동 실행 중간에 사용자의 승인을 받는 구조입니다.

- 리서치 결과를 먼저 보여 줍니다.
- 사용자가 보고서 작성 여부를 결정합니다.
- 승인하면 Writer 단계로 재개합니다.
- 승인하지 않으면 작업을 중단합니다.

그래프는 Writer 실행 직전에 멈추고, 멈춘 상태를 체크포인터에 저장해 둡니다. 승인하면 저장된 지점부터 이어서 실행합니다.

**왜 배우나:** 보고서를 쓰는 데는 시간과 호출 비용이 듭니다. 근거 자료가 부족하거나 방향이 틀렸다면 Writer가 일하기 전에 사람이 멈추는 편이 낫습니다.

In [ ]:
show_quiz("../assets/quizzes/4-3/q12.json")

**정리:** 리서치 결과를 보여 준 뒤 멈추고, 승인하면 Writer로 재개, 승인하지 않으면 중단합니다.

<details><summary>강사용 메모</summary>

- 정답 코드: `builder.compile(checkpointer=MemorySaver(), interrupt_before=['writer'])` → `await app.ainvoke(initial_state, config)` → `input('보고서 작성을 시작할까요? (y/n): ')` → `y`면 `await app.ainvoke(None, config)`로 재개, 아니면 `취소됐습니다.` 출력 후 종료.
- 터미널에서 실행하는 프로그램이라 `input()`이 그대로 동작합니다(사외 OpenAI로 y/n 모두 실행 확인, 2026-09-28).

</details>

## 2. MCP와 RAGaaS 활용 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-3", [5])

Researcher는 자료를 세 곳에서 모읍니다.

- **MCP:** Researcher가 로컬 문서 도구를 쓸 수 있게 연결합니다. `data` 경로에 있는 파일을 제공하도록 MCP 서버를 구현합니다. 문서 목록 보기, 문서 읽기, 키워드 검색 도구 세 개가 서버에 있습니다.
- **RAGaaS:** 사내 지식베이스 검색을 API로 활용합니다. 사내망에서는 RAGaaS API를 호출합니다.
- **웹 검색:** 최신 정보를 보완합니다.

Researcher는 웹 검색과 내부 문서 검색을 함께 사용합니다.

**왜 배우나:** MCP 서버를 따로 띄워 두면 문서 도구를 Agent 코드 밖에서 관리할 수 있습니다. Agent 쪽에서는 서버에 연결해 도구 목록을 받아 오기만 하면 됩니다.

In [ ]:
show_quiz("../assets/quizzes/4-3/q2.json")

**정리:** Researcher는 MCP 서버의 문서 도구, 사내 RAGaaS, 웹 검색을 함께 써서 자료를 모읍니다.

<details><summary>강사용 메모</summary>

- MCP 서버는 SSE 방식, `127.0.0.1:8001`입니다(2026-09-29 리뷰로 `0.0.0.0`에서 바꿈. 원본 정답 서버는 `0.0.0.0`). 3-3 A2A 서버도 8001을 씁니다. 포트는 그대로 두고 "앞 서버를 Ctrl+C로 끄기"로 안내합니다(사용자 결정).
- 사외(`COURSE_ENV=out`)에서는 RAGaaS가 모의 응답(`[MOCK RAGaaS] 질문: …`)을 돌려줍니다. 모의 문장에 "보험 약관"이라는 말이 들어 있어 리서치 결과에 섞일 수 있습니다.
- 사내 `index_name` 기본값(`rp-aicoa-meritz-pet`, 펫보험 약관)은 캡스톤 주제와 맞지 않습니다. 캡스톤용 사내 인덱스가 있는지 강의장에서 확인이 필요합니다(미확인).

</details>

## 3. Deep Agents와 Langfuse <small>(슬라이드) (퀴즈)</small>

### 3.1 Deep Agents <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-3", [6])

Deep Agents는 Researcher와 Writer의 Harness 역할을 맡습니다.

- `create_deep_agent`로 전문 Agent를 생성합니다.
- `FilesystemBackend`로 Skill 파일에 접근합니다.
- Researcher Skill과 Writer Skill을 분리합니다.
- MemorySaver로 Agent별 실행 상태를 유지합니다.

Researcher Skill에는 도구를 쓰는 순서와 리서치 결과 형식이, Writer Skill에는 보고서 양식이 들어 있습니다. Agent마다 자기 일에 필요한 규칙만 읽습니다.

**왜 배우나:** 규칙을 코드가 아니라 Skill 문서로 두면, 보고서 양식을 바꿀 때 코드를 고치지 않고 문서만 고치면 됩니다. 역할마다 규칙을 나누면 Agent가 남의 일까지 하려는 일도 줄어듭니다.

In [ ]:
show_quiz("../assets/quizzes/4-3/q31.json")

**정리:** `create_deep_agent`로 Agent를 만들고, `FilesystemBackend`로 Skill 파일을 읽게 하며, Researcher와 Writer의 Skill을 나눕니다.

<details><summary>강사용 메모</summary>

- **원본 정답의 조용한 결함:** 원본 정답은 `skills=['skills/researcher/']`처럼 SKILL.md가 바로 든 폴더를 가리킵니다. 설치된 deepagents 0.7.18은 이 경로를 "skill 폴더들이 든 상위 폴더"로 읽기 때문에 skill을 하나도 찾지 못합니다. 원본 정답을 그대로 실행하면 오류 없이 돌지만 Writer가 `[SDS-AI-RPT v2.1]` 양식을 따르지 않습니다(2026-09-28 사외 실행으로 확인).
- `capstone/` 폴더는 skill을 `skills/researcher/research/SKILL.md`, `skills/writer/sds-report-writer/SKILL.md`로 한 단계 더 넣고, Writer의 경로를 `skills=['skills/writer/']`로 바꿨습니다. 이렇게 하면 리서치 결과가 `## 리서치 결과:` 형식, 보고서가 `[SDS-AI-RPT v2.1]` 양식으로 나옵니다(실행 확인). 원본 `code/solution`은 고치지 않았습니다.
- `FilesystemBackend(root_dir=capstone 폴더)`라 Deep Agent의 기본 파일 도구로 이 폴더의 파일을 읽고 쓸 수 있습니다.

</details>

### 3.2 Langfuse <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-3", [7])

Langfuse로 캡스톤 실행 흐름과 평가 결과를 관측합니다.

- CallbackHandler로 모델 호출을 추적합니다.
- Researcher와 Writer 실행을 Trace로 기록합니다.
- LLM Judge 평가도 별도 Trace로 기록합니다.
- 평가 점수를 Langfuse Score로 저장합니다.

**왜 배우나:** Researcher가 어떤 도구를 몇 번 썼는지, 어느 단계에서 시간이 오래 걸렸는지는 최종 보고서만 봐서는 알 수 없습니다. Trace를 보면 실행 과정을 단계별로 되짚을 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/4-3/q32.json")

**정리:** Researcher·Writer 실행과 LLM Judge 평가가 각각 Trace로 남고, 평가 점수는 Score로 저장됩니다.

<details><summary>강사용 메모</summary>

- 사외 실행(2026-09-28) 기준 Langfuse에는 Researcher·Writer가 각각 이름 `LangGraph`인 trace로, Judge가 `RunnableSequence` trace로 남았습니다. 원본 코드는 trace 이름·태그를 따로 붙이지 않으므로 "실행한 시각 + 이름"으로 찾습니다.
- Researcher trace의 도구 호출(`list_docs`·`read_doc`·`rag_search`·`search_web`·skill을 읽는 `read_file`)을 확인했습니다. Score 네 개가 대시보드에 보이는지는 Langfuse 조회 API 제한(새 조직은 레거시 API 사용 불가)으로 자동 확인하지 못했습니다(미확인, 화면으로 확인 필요).
- Supervisor의 호출에는 CallbackHandler가 붙어 있지 않아 trace로 남지 않습니다.

</details>

## 4. 보고서 품질 평가 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("4-3", [8, 9])

최종 보고서는 LLM Judge가 네 항목으로 평가합니다.

| 항목 | 보는 것 |
|---|---|
| Relevance | 주제와 관련 있는가 |
| Faithfulness | 리서치 근거에 충실한가 |
| Completeness | 보고서 구성이 완성되어 있는가 |
| Clarity | 문장과 구조가 명확한가 |

평가 결과는 사람이 읽는 코멘트와 점수로 함께 남깁니다.

- 각 항목 0.0~1.0 점수
- 항목별 채점 근거
- 평균 점수
- Langfuse Score 기록

**왜 배우나:** 점수만 있으면 무엇을 고쳐야 할지 모릅니다. 항목별 근거가 함께 있어야 Skill 규칙이나 지시를 어떻게 바꿀지 정할 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/4-3/q4.json")

**정리:** Relevance·Faithfulness·Completeness·Clarity 네 항목을 0.0~1.0으로 채점하고, 근거와 평균 점수를 함께 남겨 Langfuse Score로 기록합니다.

<details><summary>강사용 메모</summary>

- 정답 코드의 Judge 지시에는 "0.1 단위로 채점, 1.0은 완벽할 때만"이 들어 있습니다. 사외 실행 두 번의 종합 점수는 0.90, 0.93이었습니다(점수는 실행마다 달라집니다).
- 출력 화면의 한글 이름: 관련성(Relevance)·근거충실도(Faithfulness)·완성도(Completeness)·명확성(Clarity).

</details>

## 5. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/4-3/final.json")

**이 노트북에서 본 것**
- 캡스톤은 주제 입력 → Supervisor 결정 → Researcher 자료 수집 → 사용자 승인 → Writer 초안 → 평가·출력의 여섯 단계입니다.
- Human-in-the-loop로 Writer 실행 전에 멈추고, 승인하면 재개, 승인하지 않으면 중단합니다.
- Researcher는 MCP 문서 도구, RAGaaS, 웹 검색을 함께 씁니다.
- Deep Agents로 Researcher와 Writer를 만들고, Skill을 역할별로 나눕니다.
- Langfuse에 실행 Trace와 평가 Score가 남습니다.
- LLM Judge는 Relevance·Faithfulness·Completeness·Clarity를 0.0~1.0으로 채점하고 근거를 남깁니다.